# Run Ollama in Colab
---

[![5aharsh/collama](https://raw.githubusercontent.com/5aharsh/collama/main/assets/banner.png)](https://github.com/5aharsh/collama)

This is an example notebook which demonstrates how to run Ollama inside a Colab instance. With this you can run pretty much any small to medium sized models offerred by Ollama for free.

For the list of available models check [models being offerred by Ollama](https://ollama.com/library).


## Before you proceed
---

Since by default the runtime type of Colab instance is CPU based, in order to use LLM models make sure to change your runtime type to T4 GPU (or better if you're a paid Colab user). This can be done by going to **Runtime > Change runtime type**.

While running your script be mindful of the resources you're using. This can be tracked at **Runtime > View resources**.

## Running the notebook
---

After configuring the runtime just run it with **Runtime > Run all**. And you can start tinkering around. This example uses [Llama 3.2](https://ollama.com/library/llama3.2) to generate a response from a prompted question using [LangChain Ollama Integration](https://python.langchain.com/docs/integrations/chat/ollama/).

## Installing Dependencies
---

1. `pciutils` is required by Ollama to detect the GPU type.
2. Installation of Ollama in the runtime instance will be taken care by `curl -fsSL https://ollama.com/install.sh | sh`




In [3]:
!sudo apt update
!sudo apt install -y pciutils
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
107 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as r

## Running Ollama
---

In order to use Ollama it needs to run as a service in background parallel to your scripts. Becasue Jupyter Notebooks is built to run code blocks in sequence this make it difficult to run two blocks at the same time. As a workaround we will create a service using subprocess in Python so it doesn't block any cell from running.

Service can be started by command `ollama serve`.

`time.sleep(5)` adds some delay to get the Ollama service up before downloading the model.

In [4]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

## Pulling Model
---

Download the LLM model using `ollama pull llama3.2`.

For other models check https://ollama.com/library

In [8]:
!ollama pull qwen3

## And that's it!
---

With this you should be able to freely play around with the models in your scripts. Following is an example using `langchain-ollama` to answer a simple prompt.

If you have a use-case that can help out others feel free to add your notebook to [Collama](https://github.com/5aharsh/collama/fork)

In [6]:
!pip install langchain-ollama

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from IPython.display import Markdown

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

model = OllamaLLM(model="qwen3")

chain = prompt | model

display(Markdown(chain.invoke({"question": "What's the length of hypotenuse in a right angled triangle"})))

The length of the hypotenuse in a right-angled triangle can be determined using the **Pythagorean theorem**, which states:

$$
c = \sqrt{a^2 + b^2}
$$

**Explanation:**
- $ c $ is the length of the hypotenuse (the side opposite the right angle).
- $ a $ and $ b $ are the lengths of the other two sides (the legs of the triangle).

**Steps:**
1. Square the lengths of the two legs ($ a^2 $ and $ b^2 $).
2. Add the squared values ($ a^2 + b^2 $).
3. Take the square root of the sum to find $ c $.

**Example:**
If the legs are 3 and 4:
$$
c = \sqrt{3^2 + 4^2} = \sqrt{9 + 16} = \sqrt{25} = 5
$$

**Note:** Without specific values for the legs ($ a $ and $ b $), the hypotenuse cannot be calculated numerically. The formula $ c = \sqrt{a^2 + b^2} $ is the general solution.

In [ ]:
import ollama
import json, re

def generate_bio_ollama(prompt):
    # Ollama handles the chat template and generation for you
    response = ollama.chat(
        model='qwen3',
        messages=[{'role': 'user', 'content': prompt + " /no_think"}],
        options={
            'temperature': 0,      # Equivalent to do_sample=False
            'num_predict': 1024,   # Equivalent to max_new_tokens
        }
    )

    # Extract just the text content
    content = response['message']['content']

    # Clean out any leftover reasoning tags if present
    return re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()

In [17]:
hi_bio = generate_bio_ollama("शी जिनपिंग की जीवनी लिखें।")
# "Write a biography of Xi Jinping."
print("\nHINDI:\n", hi_bio)


HINDI:
 **शी जिनपिंग की जीवनी**  

**जन्म और प्रारंभिक जीवन**  
शी जिनपिंग (जन्म: 15 सितंबर 1953) चीन के राष्ट्रपति, अखिल चीनी कम्युनिस्ट पार्टी (CCP) के अध्यक्ष और चीनी सेना के अध्यक्ष हैं। उनका जन्म शान्सी प्रांत के लियाओंग जिला में हुआ था। उनके पिता शी जिनशांग एक राजनीतिक नेता थे, जो चीन के विद्रोही आंदोलन में भाग लेते रहे। शी के पिता के बचपन में उनके परिवार को ब्रिटिश औपनिवेशिक शासन के दबाव से बचाने के लिए शान्सी में बसाया गया था।  

**शिक्षा और राजनीतिक शुरुआत**  
शी ने शान्सी विश


In [15]:
!ollama ps

NAME            ID              SIZE      PROCESSOR    CONTEXT    UNTIL               
qwen3:latest    500a1f067a9f    6.0 GB    100% GPU     4096       29 seconds from now    
